In [1]:
import json
import os
import  numpy as np
import pandas as pd
from keras_preprocessing.image import img_to_array
from multiprocess.pool import worker
from sympy.physics.units import degrees
from tensorflow.keras import Model
from tensorflow.keras.utils import Sequence,load_img
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.python.ops.numpy_ops.np_array_ops import fliplr

In [2]:
test_img="./arcade/stenosis/test/images/"
train_img="./arcade/stenosis/train/images/"
val_img="./arcade/stenosis/val/images/"

In [3]:
js_train="./arcade/stenosis/train/annotations/train.json"
js_val="./arcade/stenosis/val/annotations/val.json"
js_test="./arcade/stenosis/test/annotations/test.json"

with open (js_train,"r") as f:
    js_tra=json.load(f)

with open (js_val,"r") as b:
    js_v=json.load(b)

with open (js_test,"r") as c:
    js_te=json.load(c)

In [4]:
print(f" count train:{len(os.listdir(train_img))}")
print(f" count val:{len(os.listdir(val_img))}")
print(f" count test:{len(os.listdir(test_img))}")

 count train:1000
 count val:200
 count test:300


In [5]:
import shutil
import os
import json
from pathlib import Path

if os.path.exists("./yolo_dataset/labels"):
    shutil.rmtree("./yolo_dataset/labels")
    print("labels folder deleted")

BASE_DIR="./arcade/stenosis"
OUTPUT_DIR="./yolo_dataset"

for split in ['train', 'val', 'test']:
    os.makedirs(f'{OUTPUT_DIR}/labels/{split}', exist_ok=True)

for split in ['train', 'val', 'test']:
    json_path=f"{BASE_DIR}/{split}/annotations/{split}.json"

    print(f"\nProcessing {split}...")

    with open(json_path, 'r') as f:
        data=json.load(f)

    image_dict={img['id']: img for img in data['images']}

    annotations_by_image={}
    for ann in data['annotations']:
        img_id=ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id]=[]
        annotations_by_image[img_id].append(ann)

    for img_id, annotations in annotations_by_image.items():
        img_info=image_dict[img_id]
        file_name=img_info['file_name']
        width=img_info['width']
        height=img_info['height']

        label_file=f"{OUTPUT_DIR}/labels/{split}/{Path(file_name).stem}.txt"
        with open(label_file, 'w') as f:
            for ann in annotations:
                category_id=ann['category_id'] - 1
                segmentation=ann['segmentation'][0]
                seg_normalized=[]
                for i in range(0, len(segmentation), 2):
                    x=segmentation[i]/width
                    y=segmentation[i+1]/height
                    seg_normalized.extend([x, y])
                seg_str=' '.join([f"{coord:.6f}" for coord in seg_normalized])
                f.write(f"{category_id} {seg_str}\n")

    print(f"{split} done")

print("\n Conversion completed!")

labels folder deleted

Processing train...
train done

Processing val...
val done

Processing test...
test done

 Conversion completed!


In [6]:
from ultralytics import YOLO
model=YOLO("yolov8m-seg.pt")

In [7]:
print(type(train_img))

<class 'str'>


In [8]:
import os

print(len(os.listdir("./yolo_dataset/images/train")))
print(len(os.listdir("./yolo_dataset/labels/train")))

997
997


In [9]:
model.train(data="dataset.yaml",epochs=200,augment=True,hsv_h=0.0,hsv_s=0.0,hsv_v=0.2,degrees=15.0,translate=0.1,flipud=0.5,fliplr=0.5,batch=8,save=True,imgsz=512,workers=2,mixup=0.0,mosaic=0.0)
history=model.val()

New https://pypi.org/project/ultralytics/8.4.71 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.70  Python-3.10.6 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale

In [10]:
print(f"box map50: {history.box.map50:.4f}")
print(f"box map50-95: {history.box.map:.4f}")
print(f"box precision: {history.box.mp:.4f}")
print(f"box recall: {history.box.mr:.4f}")

box map50: 0.3476
box map50-95: 0.1379
box precision: 0.4044
box recall: 0.3842


In [11]:
print(f"mask map50: {history.seg.map50:.4f}")
print(f"mask map50-95: {history.seg.map:.4f}")
print(f"precision: {history.seg.mp:.4f}")
print(f"recall: {history.seg.mr:.4f}")

mask map50: 0.4049
mask map50-95: 0.1482
precision: 0.4429
recall: 0.4368
